# Other matchers at height, with a bar set per matcher from its own wrong-place counts (iteration 36)

**Why.** LoFTR proposes almost no matches above 1400 m (clumps of a few pixels, 8-pixel patches). Other MINIMA matchers are on disk: XoFTR (LoFTR's cross-modality successor) and RoMa (dense, DINOv2 encoder). E-LoFTR needs code we do not have; LightGlue is sparse and needs a download.

**The bar problem (user, 2026-11-18).** Inlier counts are on different scales: LoFTR counts a few tens at the true place, RoMa samples thousands of dense matches and counts hundreds even at wrong places. So a bar of 8 means nothing across matchers. Each matcher gets its **own bar from its own wrong-place counts**: for every photo the same canvases are matched against up to 4 WRONG tiles (3 km east / west / north / south, where the map exists); the null sample is the most inliers over the queries on each wrong tile. The bar is the largest null count (zero false positives) or its 99th percentile; recall = share of photos whose true-place count (most inliers over the queries) exceeds it. To avoid setting the bar and judging on the same photos, the bar is calibrated on one half of the photos (odd index) and the recall measured on the other half (even index), and the other way round. A threshold-free number is also given: the AUC of true-place counts against the null counts.

**Inputs.** `normal` = the pipeline's 224 px canvas and tile; `native cleaned` = 448 px canvas two levels finer, photo cleaned, tile locally normalised (iteration 34). 3 queries per photo (headings -5, 0, +5 at scale 1.0), development photos above 800 m only.

**Kill rule (written before the run).** A matcher is useful only if its cross-validated recall at the zero-false-positive bar is at least 10 points above LoFTR's with the same input, in at least two of the three height bands (800-1100, 1100-1400, 1400-2100 m), AND AUC at least 0.9 above 1400 m. The seconds per pair are reported (Jetson budget: the 30 s limit per photo is for a machine about 3x slower than this server).

Env: `EF_HEADLESS, EF_DATASET, EF_STORE, EF_OUT, EF_TAG, EF_LIMIT, EF_MATCHERS, EF_EVERY` (EF_EVERY=2 uses every second photo).

In [ ]:
import os, sys, math, colorsys
import numpy as np, pandas as pd, torch, cv2
HEADLESS = os.environ.get("EF_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("EF_LEVELS", "3,4,5,6,7,8,9,10,11").split(",")]
from geoloc import photo, queries, scoring, fine_matching, flat_field
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query, pick_peaks, window_transforms, zncc_everywhere, fast_length, cut_map_features
from geoloc.edge_features import normalise_cells
from geoloc.levels import level_for_height, level_gsd, map_piece
from geoloc.map_store import MapStore
CELL = settings.CELL; TILE = settings.TILE
DATASET = os.environ["EF_DATASET"]; STORE = os.environ["EF_STORE"]; OUT = os.environ.get("EF_OUT", "results/edge_features"); os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE); rows = read_rows(DATASET)       # the dataset folder holds the development photos only
heights = np.array([float(r["alt"]) for r in rows]); print(len(rows), "photos, heights", heights.min(), heights.max())

In [ ]:
import time, sys
from PIL import Image
LIMIT = int(os.environ.get("EF_LIMIT", 0)); MATCHERS = os.environ.get("EF_MATCHERS", "loftr,xoftr,roma").split(",")
MODELS = os.path.join(os.path.abspath(HEADLESS), "models"); CKPT = os.environ.get("EF_CKPT", "/mnt/e/IdeaForge/work/NUDT/checkpoint")
ORIGINAL_NORMALISE = photo.normalise_frame
engines = {}
loftr = fine_matching.load_loftr(os.path.join(MODELS, settings.LOFTR_FILE))
def loftr_pairs(a, b):
    pa, pb, conf = fine_matching.loftr_matches(loftr, a, b); keep = conf >= settings.LOFTR_CONFIDENCE; return pa[keep], pb[keep]
engines["loftr"] = loftr_pairs
if "xoftr" in MATCHERS:
    THIRD = os.environ.get("EF_XOFTR", "/mnt/e/IdeaForge/work/repo/third_party/XoFTR"); sys.path.insert(0, THIRD)
    from xoftr import XoFTR
    def xoftr_config(coarse, fine):
        return dict(resolution=(8, 2), fine_window_size=5, medium_window_size=3, resnet=dict(initial_dim=128, block_dims=[128, 196, 256]),
                    coarse=dict(inference=True, d_model=256, d_ffn=256, nhead=8, layer_names=["self", "cross"] * 4, attention="linear"),
                    match_coarse=dict(inference=True, d_model=256, thr=coarse, border_rm=2, match_type="dual_softmax", dsmax_temperature=0.1, train_coarse_percent=0.2, train_pad_num_gt_min=200),
                    fine=dict(denser=False, inference=True, dsmax_temperature=0.1, thr=fine, mlp_hidden_dim_coef=2, nhead_fine_level=8, nhead_medium_level=7))
    saved = torch.load(os.path.join(CKPT, "MINIMA", "minima_xoftr.ckpt"), map_location="cpu", weights_only=False)
    weights = {k[len("matcher."):]: v for k, v in saved["state_dict"].items() if k.startswith("matcher.")}
    xmodel = XoFTR(config=xoftr_config(0.3, 0.1)); xmodel.load_state_dict(weights, strict=True); xmodel = xmodel.to(DEVICE).eval()
    @torch.no_grad()
    def xoftr_pairs(a, b):
        batch = {"image0": fine_matching.as_tensor(a), "image1": fine_matching.as_tensor(b)}; xmodel(batch)
        return batch["mkpts0_f"].cpu().numpy(), batch["mkpts1_f"].cpu().numpy()
    engines["xoftr"] = xoftr_pairs
if "roma" in MATCHERS:
    from romatch import roma_outdoor
    rweights = torch.load(os.path.join(CKPT, "MINIMA", "minima_roma.pth"), map_location=DEVICE); dino = torch.load(os.path.join(CKPT, "RoMa", "dinov2_vitl14_pretrain.pth"), map_location=DEVICE)
    roma = roma_outdoor(DEVICE, weights=rweights, dinov2_weights=dino, coarse_res=560, upsample_res=(672, 672), amp_dtype=torch.float16, use_custom_corr=False).eval()
    def to_pil(g): u = np.clip(np.asarray(g, np.float32) * 255, 0, 255).astype(np.uint8); return Image.fromarray(np.stack([u, u, u], -1))
    @torch.no_grad()
    def roma_pairs(a, b):
        torch.manual_seed(0); warp, cert = roma.match(to_pil(a), to_pil(b), device=DEVICE); m, _ = roma.sample(warp, cert, num=2000)
        ha, wa = a.shape; hb, wb = b.shape; pa, pb = roma.to_pixel_coordinates(m, ha, wa, hb, wb); return pa.cpu().numpy().astype(np.float32), pb.cpu().numpy().astype(np.float32)
    engines["roma"] = roma_pairs
print("matchers:", list(engines))

In [ ]:
def with_settings(flat=None, degree=3, local=None):
    settings.FLAT_FIELD_FROM_HEIGHT_M = flat; settings.FLAT_FIELD_DEGREE = degree; settings.LOCAL_NORM_SIGMA = local
def clean_tile(tile, sigma):
    has = tile > 0; fill = float(tile[has].mean()) if has.any() else 0.5
    normal = flat_field.local_normalise(np.where(has, tile, fill).astype(np.float32), sigma); out = ORIGINAL_NORMALISE(normal)
    return np.where(has, out, float(out[has].mean()) if has.any() else 0.5).astype(np.float32)
INPUTS = [("normal", 0), ("native cleaned", 2)] if os.environ.get("EF_INPUTS", "both") == "both" else [("native cleaned", 2)]
def inliers_of(pairs):
    pa, pb = pairs
    if len(pa) < 4: return 0
    matrix, inl = fine_matching.fit_similarity(pa.astype(np.float32), pb.astype(np.float32)); return int(inl.sum()) if matrix is not None else 0

In [ ]:
def wrong_tiles(level_image, level, truth, size, limit=4):
    tiles = []
    for dx, dy in ((3000.0, 0.0), (-3000.0, 0.0), (0.0, 3000.0), (0.0, -3000.0)):
        cand = map_piece(level_image, level, truth[0] + dx, truth[1] + dy, size)
        if (cand > 0).mean() > 0.9: tiles.append(cand)
    return tiles[:limit]
query_list = [q for q in queries.query_list() if abs(q["scale_factor"] - 1.0) < 1e-6]; records = []; clock = {}
EVERY = int(os.environ.get("EF_EVERY", 1))
heights_ok = [i for i, r in enumerate(rows) if float(r["alt"]) >= 800][::EVERY]
if LIMIT: heights_ok = heights_ok[:LIMIT]
for index in heights_ok:
    meta = rows[index]; camera = photo.read_camera(meta); pose = photo.read_pose(meta); path = os.path.join(DATASET, "images", meta["filename"]); level = level_for_height(pose["height"], camera)
    truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"])); photo_gsd = pose["height"] / camera["fx"]
    for inp, drop in INPUTS:
        lf = level - drop
        if lf < min(settings.LEVELS) or lf not in store.levels: continue
        size = settings.TILE * (2 ** (drop // 2)); settings.TILE = size
        try:
            if drop: with_settings(0, 3, 15.0)
            else: with_settings(None)
            picture = photo.prepare_photo(path, pose, camera, lf); with_settings(None)
            canvases = [fine_matching.grey_canvas(*photo.draw_canvas(picture, pose, camera, lf, q["heading"], q["scale_factor"])) for q in query_list]
            true_tile = map_piece(store.levels[lf], lf, truth[0], truth[1], size); bad = wrong_tiles(store.levels[lf], lf, truth, size)
            if len(bad) == 0: continue
            if drop:
                sigma_tile = max(3.0, 15.0 * photo_gsd / level_gsd(lf)); true_tile = clean_tile(true_tile, sigma_tile); bad = [clean_tile(t, sigma_tile) for t in bad]
        finally:
            settings.TILE = 224
        for name, engine in engines.items():
            t0 = time.time(); calls = 0; true_best = 0; null_best = [0] * len(bad)
            for c in canvases:
                true_best = max(true_best, inliers_of(engine(c, true_tile))); calls += 1
                for k, tile in enumerate(bad): null_best[k] = max(null_best[k], inliers_of(engine(c, tile))); calls += 1
            if DEVICE == "cuda": torch.cuda.synchronize()
            clock.setdefault((name, inp), []).append((time.time() - t0) / calls)
            records.append(dict(photo=index, height=pose["height"], input=inp, matcher=name, true_inl=true_best, null_inl=null_best))
    print(index, flush=True)
tab = pd.DataFrame(records); tab.to_pickle(os.path.join(OUT, "matchers_at_height%s.pkl" % os.environ.get("EF_TAG", "")))
pd.DataFrame([dict(matcher=k[0], input=k[1], seconds_per_pair=float(np.mean(v))) for k, v in clock.items()]).to_csv(os.path.join(OUT, "matchers_cost%s.csv" % os.environ.get("EF_TAG", "")), index=False)
print(len(tab), "rows saved")

In [ ]:
from sklearn.metrics import roc_auc_score
tab["band"] = pd.cut(tab.height, [800, 1100, 1400, 2100]); tab["fold"] = tab.photo % 2
def cv_recall(g, quantile):
    # bar from the other fold's null counts, recall on this fold's true counts
    hits = []
    for fold in (0, 1):
        nulls = np.concatenate([np.array(v, float) for v in g[g.fold != fold].null_inl]); bar = nulls.max() if quantile is None else np.quantile(nulls, quantile)
        sel = g[g.fold == fold]; hits += list((sel.true_inl > bar).astype(float))
    return float(np.mean(hits))
rows_out = []
for (m, i), g in tab.groupby(["matcher", "input"]):
    nulls = np.concatenate([np.array(v, float) for v in g.null_inl]); t = g.true_inl.values.astype(float)
    auc = roc_auc_score(np.r_[np.ones(len(t)), np.zeros(len(nulls))], np.r_[t, nulls])
    rows_out.append(dict(matcher=m, input=i, photos=len(g), true_median=np.median(t), null_median=np.median(nulls), null_max=nulls.max(), null_q99=np.quantile(nulls, 0.99), auc=round(auc, 3),
                         recall_zero_fp_cv=round(cv_recall(g, None), 3), recall_q99_cv=round(cv_recall(g, 0.99), 3), recall_in_sample=round(float((t > nulls.max()).mean()), 3)))
cost = pd.read_csv(os.path.join(OUT, "matchers_cost%s.csv" % os.environ.get("EF_TAG", "")))
print("above 800 m: bars per matcher from its own wrong-place counts"); print(pd.DataFrame(rows_out).merge(cost, on=["matcher", "input"]).round(3).to_string(index=False))
print("\ncross-validated recall at the zero-false-positive bar, by height band")
band_rows = []
for (b, m, i), g in tab.groupby(["band", "matcher", "input"], observed=True):
    nulls_all = tab[(tab.matcher == m) & (tab.input == i)]
    hits = []
    for fold in (0, 1):
        nulls = np.concatenate([np.array(v, float) for v in nulls_all[nulls_all.fold != fold].null_inl]); bar = nulls.max()
        hits += list((g[g.fold == fold].true_inl > bar).astype(float))
    ba = roc_auc_score(np.r_[np.ones(len(g)), np.zeros(sum(len(v) for v in g.null_inl))], np.r_[g.true_inl.values.astype(float), np.concatenate([np.array(v, float) for v in g.null_inl])]) if len(g) > 2 else np.nan
    band_rows.append(dict(band=str(b), matcher=m, input=i, n=len(g), recall_zero_fp_cv=round(float(np.mean(hits)), 2), auc=round(ba, 2)))
print(pd.DataFrame(band_rows).pivot_table(index="band", columns=["matcher", "input"], values=["recall_zero_fp_cv", "auc"]).round(2).to_string())